In [1]:
# toy_transformer_reverse + learning eos token
# -----------------------------------------------
# A minimal, easy-to-read Transformer in PyTorch
# that learns to reverse a short character string.
# Focuses on clarity, not speed.63+++

import math, random, string, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ------------------------------
# 1 Dataset: reverse the string
# ------------------------------

ALPHABET   = string.ascii_lowercase   # 26 letters
PAD_TOKEN  = "<pad>"
SOS_TOKEN  = "<s>"                    # start-of-sequence for decoder-like LM
EOS_TOKEN  = "_"
ALL_TOKENS = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN] + list(ALPHABET)
VOCAB_SIZE = len(ALL_TOKENS)
CHAR2IDX   = {ch: i for i, ch in enumerate(ALL_TOKENS)}
IDX2CHAR   = {i: ch for ch, i in CHAR2IDX.items()}

MAX_LEN    = 16                       # keep sequences short for clarity

def encode_inp(seq: str) -> list[int]:
    """String → list of indices incl. <s> at front, padded to MAX_LEN+1."""
    seq = seq.lower()
    assert len(seq) <= MAX_LEN, "too long"
    idxs        = [CHAR2IDX[SOS_TOKEN]] + [CHAR2IDX[c] for c in seq]
    pad_needed  = (MAX_LEN + 2) - len(idxs) # pad to max length
    idxs       += [CHAR2IDX[PAD_TOKEN]] * pad_needed
    return idxs  # length MAX_LEN+2

def encode_tgt(seq: str) -> list[int]:
    """String → list of indices incl. <s> at front, padded to MAX_LEN+1.<e> at back"""
    seq = seq.lower()
    assert len(seq) <= MAX_LEN, "too long"
    idxs        = [CHAR2IDX[SOS_TOKEN]] + [CHAR2IDX[c] for c in seq] + [CHAR2IDX[EOS_TOKEN]]
    pad_needed  = (MAX_LEN + 2) - len(idxs) # pad to max length
    idxs       += [CHAR2IDX[PAD_TOKEN]] * pad_needed
    return idxs  # length MAX_LEN+2

def decode(idxs: list[int]) -> str:
    """Drop SOS & PAD and turn back into string."""
    return "".join(IDX2CHAR[i] for i in idxs if i > 1)

class ReverseDataset(Dataset):
    """Generate random strings on-the-fly; split = 'train' or 'val'."""
    def __init__(self, split="train", n_samples=10_000):
        random.seed(0 if split=="train" else 1)
        self.samples = [
            "".join(random.choices(ALPHABET, k=random.randint(3, MAX_LEN)))
            for _ in range(n_samples)
        ]
    
    def __len__(self):  return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        inp = torch.tensor(encode_inp(s), dtype=torch.long)
        tgt = torch.tensor(encode_tgt(s[::-1]), dtype=torch.long)  # reversed
        return inp, tgt

train_loader = DataLoader(ReverseDataset("train"), batch_size=64, shuffle=True)
val_loader   = DataLoader(ReverseDataset("val", n_samples=2000), batch_size=64)

# ------------------------------
# 2 Positional encoding (sinusoid)
# ------------------------------

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=1024):  # increase max_len
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        self.pe      = torch.zeros(max_len, d_model)
        position     = torch.arange(0, max_len).unsqueeze(1)
        div_term     = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        
        self.pe[:, 0::2] = torch.sin(position * div_term)
        self.pe[:, 1::2] = torch.cos(position * div_term)
        self.pe          = self.pe.unsqueeze(1)

    def forward(self, x):
        if x.size(0) > self.pe.size(0):
            raise ValueError(f"Input sequence length {x.size(0)} exceeds positional encoding max_len {self.pe.size(0)}")
        x = x + self.pe[:x.size(0)]
        return self.dropout(x)

# ------------------------------
# 3 One “post-norm” Transformer block
# ------------------------------

class MiniTransformerLayer(nn.Module):
    """
    Post-norm variant:

        y = LN(x +  MHA(LN(x)) )
        z = LN(y +  FFN(LN(y)) )
    """
    def __init__(self, d_model=128, num_heads=4, d_ff=256, p_drop=0.1):
        super().__init__()
        self.mha = nn.MultiheadAttention(d_model, num_heads, dropout=p_drop, batch_first=True)
        self.ff  = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(p_drop)

    def forward(self, x, padding_mask=None):
        # --- block 1: self-attention residual ---
        x_norm = self.ln1(x)
        attn_out, _ = self.mha(x_norm, x_norm, x_norm,
                               key_padding_mask=padding_mask)
        x = x + self.drop(attn_out)

        # --- block 2: feed-forward residual ---
        y_norm = self.ln2(x)
        ff_out = self.ff(y_norm)
        x = x + self.drop(ff_out)
        return x

# ------------------------------
# 4 Full tiny Transformer encoder
# ------------------------------

class TinyTransformer(nn.Module):
    def __init__(self,
                 vocab_size=VOCAB_SIZE,
                 d_model=128,
                 n_layers=2,
                 num_heads=4,
                 d_ff=256):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos   = PositionalEncoding(d_model)
        self.layers = nn.ModuleList(
            [MiniTransformerLayer(d_model, num_heads, d_ff) for _ in range(n_layers)]
        )
        self.proj = nn.Linear(d_model, vocab_size)

    def forward(self, tok, padding_mask=None):
        """
        tok : (batch, seq)  integers
        padding_mask : (batch, seq) True on PAD
        """
        x = self.embed(tok)                 # (batch, seq, d_model)
        x = self.pos(x.transpose(0,1))      # → (seq, batch, d_model)
        x = x.transpose(0,1)                # back to (batch, seq, d_model)

        for layer in self.layers:
            x = layer(x, padding_mask)

        return self.proj(x)                 # logits (batch, seq, vocab)

model = TinyTransformer()
print("Model parameters:", sum(p.numel() for p in model.parameters()))

# ------------------------------
# 5 Training helpers
# ------------------------------

def make_padding_mask(batch_tok: torch.Tensor) -> torch.Tensor:
    """True where PAD so MultiheadAttention can ignore them."""
    return batch_tok.eq(CHAR2IDX[PAD_TOKEN])

# criterion = nn.CrossEntropyLoss(ignore_index=CHAR2IDX[PAD_TOKEN])
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ------------------------------
# 6 Train loop
# ------------------------------

def run_epoch(loader, train=True):
    model.train(mode=train)
    total, correct = 0, 0
    total_loss = 0
    
    for inp, tgt in loader:
        mask   = make_padding_mask(inp)
        logits = model(inp, padding_mask=mask)          # (batch, seq, vocab)
        loss   = criterion(logits.view(-1, VOCAB_SIZE), tgt.view(-1))

        if train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # Accumulate loss for reporting
        total_loss += loss.item()
        
        # crude accuracy (ignoring PAD & SOS)
        with torch.no_grad():
            preds    = logits.argmax(-1)
            valid    = tgt.ne(CHAR2IDX[PAD_TOKEN])
            correct += (preds.eq(tgt) & valid).sum().item()
            total   += valid.sum().item()

    avg_loss = total_loss / len(loader)
    
    return avg_loss, correct / total

for epoch in range(10):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss,   val_acc   = run_epoch(val_loader,   train=False)
    print(f"Epoch {epoch:2d} | "
          f"train loss {train_loss:.3f}, acc {train_acc:.2%} | "
          f"val loss {val_loss:.3f}, acc {val_acc:.2%}")

    # ---- qualitative sample block (5 random val samples) ----
    if epoch % 1 == 0:
        sample_batch = next(iter(DataLoader(ReverseDataset("val", n_samples=128), batch_size=8, shuffle=True)))
        sample_inp, sample_tgt = sample_batch
        pred = model(sample_inp, make_padding_mask(sample_inp)).argmax(-1)
        # pred = generate_until_eos(model, sample_inp)
    
        print(" Example predictions:")
        for i in range(min(5, len(sample_inp))):
            print(f"   input : {decode(sample_inp[i].tolist())}")
            print(f"   target: {decode(sample_tgt[i].tolist())}")
            print(f"   pred  : {decode(pred[i].tolist())}")
            print("   " + "-"*30)


# -----------------------------------------------
# End of script
# -----------------------------------------------

Model parameters: 272413
Epoch  0 | train loss 1.523, acc 27.08% | val loss 0.966, acc 45.12%
 Example predictions:
   input : owkaahpcfrz
   target: zrfcphaakwo_
   pred  : ffffaaaaaaa_
   ------------------------------
   input : oubdt
   target: tdbuo_
   pred  : ttdto_
   ------------------------------
   input : goltvfhskdfopy
   target: ypofdkshfvtlog_
   pred  : yyffffffffooog_
   ------------------------------
   input : npazgg
   target: ggzapn_
   pred  : gggann_
   ------------------------------
   input : plhoyauvxtvnolbw
   target: wblonvtxvuayohlp_
   pred  : llllvxxvvvllllll
   ------------------------------
Epoch  1 | train loss 0.994, acc 46.25% | val loss 0.547, acc 67.11%
 Example predictions:
   input : nefloghnmkcj
   target: jckmnhgolfen_
   pred  : jjccnnhglfee_
   ------------------------------
   input : kqihnpozeqz
   target: zqezopnhiqk_
   pred  : zqzzopnhqkk_
   ------------------------------
   input : ywdbsksgqxx
   target: xxqgsksbdwy_
   pred  : xxqqsss